In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
from torchvision.models import VGG16_Weights

In [2]:
!pip install pyyaml


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
!pip install kagglehub

  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)

   ---------------------------------------- 0/4 [tqdm]
   ---------------------------------------- 0/4 [tqdm]
   ---------- ----------------------------- 1/4 [protobuf]
   ---------- ----------------------------- 1/4 [protobuf]
   ---------- ----------------------------- 1/4 [protobuf]
   ---------- ----------------------------- 1/4 [protobuf]
   ---------- ----------------------------- 1/4 [protobuf]
   ---------- ----------------------------- 1/4 [protobuf]
   ---------- ----------------------------- 1/4 [protobuf]
   -------------------- ------------------- 2/4 [kagglesdk]
   -------------------- ------------------- 2/4 [kagglesdk]
   -------------------- ------------------- 2/4 [kagglesdk]
   -------------------- ------------------- 2/4 [kagglesdk]
   -------------------- ------------------- 2/4 [kagglesdk]
   -------------------- ------------------- 2/4 [kagglesdk]
   --


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ultralytics/coco128")

print("Path to dataset files:", path)

c:\Users\arthu\FastCrowdVision\my_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\arthu\.cache\kagglehub\datasets\ultralytics\coco128\versions\3


In [3]:
import os

dataset_root = "C:/Users/arthu/.cache/kagglehub/datasets/ultralytics/coco128/versions/3"

for root, dirs, files in os.walk(dataset_root):
    print("DIR:", root)
    print("Subdirs:", dirs)
    print("Files:", files[:5])
    print("-" * 50)

DIR: C:/Users/arthu/.cache/kagglehub/datasets/ultralytics/coco128/versions/3
Subdirs: ['coco128']
Files: []
--------------------------------------------------
DIR: C:/Users/arthu/.cache/kagglehub/datasets/ultralytics/coco128/versions/3\coco128
Subdirs: ['images', 'labels']
Files: ['LICENSE', 'README.txt']
--------------------------------------------------
DIR: C:/Users/arthu/.cache/kagglehub/datasets/ultralytics/coco128/versions/3\coco128\images
Subdirs: ['train2017']
Files: []
--------------------------------------------------
DIR: C:/Users/arthu/.cache/kagglehub/datasets/ultralytics/coco128/versions/3\coco128\images\train2017
Subdirs: []
Files: ['000000000009.jpg', '000000000025.jpg', '000000000030.jpg', '000000000034.jpg', '000000000036.jpg']
--------------------------------------------------
DIR: C:/Users/arthu/.cache/kagglehub/datasets/ultralytics/coco128/versions/3\coco128\labels
Subdirs: ['train2017']
Files: []
--------------------------------------------------
DIR: C:/Users/art

In [11]:
lbl_dir

'C:/Users/arthu/.cache/kagglehub/datasets/ultralytics/coco128/versions/3coco128/labels/train2017'

In [4]:
import glob

img_dir = dataset_root + "/coco128/images/train2017"
lbl_dir = dataset_root + "/coco128/labels/train2017"

images = sorted(glob.glob(img_dir + "/*.jpg"))
labels = sorted(glob.glob(lbl_dir + "/*.txt"))

print("Images:", len(images))
print("Labels:", len(labels))
print(images[0])
print(labels[0])

Images: 128
Labels: 128
C:/Users/arthu/.cache/kagglehub/datasets/ultralytics/coco128/versions/3/coco128/images/train2017\000000000009.jpg
C:/Users/arthu/.cache/kagglehub/datasets/ultralytics/coco128/versions/3/coco128/labels/train2017\000000000009.txt


In [13]:
len(images)

128

In [5]:
from dataloader import DataSSD300,DataSplitter

In [6]:
coco128loader=DataSSD300(img_dir,lbl_dir,gt_normalised=True)
splitter=DataSplitter(20,0.15,0.15)
train_dataloader, val_dataloader, test_dataloader=splitter(coco128loader)

In [7]:
vgg = models.vgg16(weights=VGG16_Weights.IMAGENET1K_V1).features
vgg[16] = nn.MaxPool2d(kernel_size=2, stride=2, ceil_mode=True)
base=nn.ModuleList(vgg[:30])#until 5_3 layer
base




ModuleList(
  (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU(inplace=True)
  (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (3): ReLU(inplace=True)
  (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (6): ReLU(inplace=True)
  (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (8): ReLU(inplace=True)
  (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (11): ReLU(inplace=True)
  (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (13): ReLU(inplace=True)
  (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (15): ReLU(inplace=True)
  (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=True)
  (17): Conv2d(256, 512, kernel_siz

In [8]:
from utils import setup_device_and_seed

device = setup_device_and_seed(seed=42)

In [10]:
from ssd import SSD

model=SSD(base,nb_classes=21,phase="train",alpha=1,device=device)
model=model.to(device)

In [11]:
from train import train



In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, weight_decay = 0.0005, momentum = 0.9)
train(model,optimizer,train_dataloader,val_dataloader,modelname="ssd_coco128V1",device=device)